# 📊 Bitcoin Market Analysis (Merged Historical + Real-Time)
This notebook combines **historical hourly** and **real-time streamed** Bitcoin data to perform a holistic analysis. We aim to uncover patterns, anomalies, and price behavior useful for:
- 📈 Trend detection (daily, hourly, minutely)
- 📉 Volatility and anomaly spotting
- 🤔 Buy/sell signal exploration

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), 'src'))

from bitcoin_utils import load_protobuf_file, protobufs_to_dataframe
import matplotlib.pyplot as plt
import pandas as pd
import datetime

## 🔄 Load and Combine Historical + Real-Time Data

In [ ]:
hist_msgs = load_protobuf_file("src/data/bitcoin_historical_hourly.pb")
hist_df = protobufs_to_dataframe(hist_msgs)

real_path = datetime.datetime.now().strftime("src/data/bitcoin_data_%Y-%m-%d.pb")
real_msgs = load_protobuf_file(real_path)
real_df = protobufs_to_dataframe(real_msgs)

full_df = pd.concat([hist_df, real_df], ignore_index=True)
full_df = full_df.drop_duplicates(subset="timestamp").sort_values("timestamp")
print(f"📦 Combined records: {len(full_df)}")
full_df.tail()

## ⏱️ Bitcoin Price - Last 24 Hours

In [ ]:
last_24h = full_df.set_index("timestamp").last("24H")
last_24h["price"].plot(figsize=(12, 4), title="Bitcoin Price (Last 24 Hours)")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.show()

## 🕒 Bitcoin Price - Last 3 Hours

In [ ]:
last_3h = full_df.set_index("timestamp").last("3H")
last_3h["price"].plot(figsize=(12, 4), title="Bitcoin Price (Last 3 Hours)")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.show()

## 📆 Daily Price Summary - Last 30 Days

In [ ]:
full_df['date'] = full_df['timestamp'].dt.date
daily_summary = full_df.groupby('date')["price"].agg(["min", "max", "mean"])
daily_summary.plot(figsize=(12, 5), title="Daily Bitcoin Price Stats")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.show()

## ⚠️ High Volatility Hours (Anomaly Candidates)

In [ ]:
hourly = full_df.copy()
hourly['hour'] = hourly['timestamp'].dt.floor('H')
volatility = hourly.groupby('hour')['price'].agg(['min', 'max'])
volatility['range'] = volatility['max'] - volatility['min']
anomalies = volatility[volatility['range'] > volatility['range'].mean() + 2*volatility['range'].std()]
anomalies.tail(10)

## 💡 Recent 15 Minute Stats (For Trade Insight)

In [ ]:
last_15min = full_df.set_index("timestamp").last("15min")
summary = last_15min["price"].agg(["min", "max", "mean"])
summary